### Converting .flv files to .mp4s and applying OpenFace

In [ ]:
import os               # VSCode Navigator
import subprocess       # Terminal Access
import pandas as pd
import opensmile        # OpenSmile

video_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/VideoFlash'
mp4_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/mp4'
output_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/output'
wav_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/AudioWAV'


os.makedirs(mp4_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)


# A list of all the files
flv_video_files = [video for video in os.listdir(video_dir) if video.endswith('.flv')][:1736]

for video in flv_video_files:

    # This converts the .flv file into mp4 files for OpenFace
    mp4_video_name = video.replace(".flv", ".mp4")
    subprocess.run([
        'ffmpeg',
        '-i',
        os.path.join(video_dir, video),
        '-y',
        '-loglevel',
        'error',
        os.path.join(mp4_dir, mp4_video_name)])
    

    # subprocess.run([
    # "docker", "run", "--rm",
    # "--platform", "linux/amd64",
    # "-v", "/Users/songye/Desktop/Dev/REU/CREMA-D:/data",
    # "algebr/openface:latest",
    # "/home/openface-build/build/bin/FeatureExtraction",
    # "-f", "/data/mp4/" + mp4_video_name,
    # "-out_dir", "/data/output/"])

    # This did not work, so have to run Docker manually via Terminal
    # docker run -it -v /Users/songye/Desktop/Dev/REU/CREMA-D:/data algebr/openface:latest
    # for f in /data/mp4/*.mp4; do /home/openface-build/build/bin/FeatureExtraction -f "$f" -out_dir /data/output/; done

    

### Instantiating OpenSmile

In [10]:
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.emobase,
    feature_level=opensmile.FeatureLevel.Functionals,
)

### Constructing the Visual Feature Matrix

In [48]:
from scipy.stats import linregress

# All 7442 video CSVs
table_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/table'

# Now we need a feature matrix
# Each row = one entire video, columns = aggregated AU statistics + audio features + emotion label (Each AU's mean and std)
list_of_csvs = [csv for csv in os.listdir(table_dir) if csv.endswith(".csv")]
rows = []

first_df = pd.read_csv(os.path.join(table_dir, list_of_csvs[0]))
first_df.columns = first_df.columns.str.strip()
cols_order = [col for col in first_df.columns 
              if col.startswith('AU') and (col.endswith('_r') or col.endswith('_c'))]

for csv in list_of_csvs:
    df = pd.read_csv(os.path.join(table_dir, csv))
    df.columns = df.columns.str.strip()
    au_cols = [col for col in df.columns 
           if col.startswith('AU') and (col.endswith('_r') or col.endswith('_c'))]

    list_means = []                   
    list_std = []
    list_max = []
    list_min = []
    list_first_quartile = []
    list_medians = []
    list_third_quartile = []
    list_argmin = []
    list_argmax = []
    list_range = []
    list_lower_iqr = []
    list_upper_iqr = []
    list_iqr = []
    list_kurtosis = []
    list_slope = []
    list_intercept = []
    list_p_value = []
    list_std_err = []
    list_r_squared = []
    list_skew = []                   

    for col in au_cols:
        list_means.append(df[col].mean())
        list_std.append(df[col].std())
        list_max.append(df[col].max())
        list_min.append(df[col].min())
        list_first_quartile.append(df[col].quantile(0.25))
        list_medians.append(df[col].median())
        list_third_quartile.append(df[col].quantile(0.75))
        list_argmin.append(df[col].idxmin())
        list_argmax.append(df[col].idxmax())
        list_range.append(df[col].max() - df[col].min())
        list_lower_iqr.append(df[col].quantile(0.5) - df[col].quantile(0.25))
        list_upper_iqr.append(df[col].quantile(0.75) - df[col].quantile(0.5))
        list_iqr.append(df[col].quantile(0.75) - df[col].quantile(0.25))
        list_kurtosis.append(df[col].kurt())
        slope, intercept, r_value, p_value, std_err = linregress(range(len(df)), df[col])
        list_slope.append(slope)
        list_intercept.append(intercept)
        list_p_value.append(p_value)
        list_std_err.append(std_err)
        list_r_squared.append(r_value ** 2)
        list_skew.append(df[col].skew())

    
    audio_csv = csv.replace(".csv", ".wav")
    audio_csv = os.path.join(wav_dir, audio_csv)
    audio_features = smile.process_file(audio_csv).values.flatten().tolist()

    # To get the emotion from the file name
    emotion = csv.split('_')[2]
    total_list = (list_means + list_std + list_max + list_min +
              list_first_quartile + list_medians + list_third_quartile +
              list_argmin + list_argmax + list_range +
              list_lower_iqr + list_upper_iqr + list_iqr +
              list_kurtosis + list_slope + list_intercept +
              list_p_value + list_std_err + list_r_squared +
              list_skew + audio_features + [emotion])

    rows.append(total_list)


sample_audio = os.path.join(wav_dir, os.listdir(wav_dir)[0])
audio_col_names = list(smile.process_file(sample_audio).columns)
col_names = ([col + '_mean' for col in cols_order] + 
             [col + '_std' for col in cols_order] + 
             [col + '_max' for col in cols_order] +
             [col + '_min' for col in cols_order] +
             [col + '_first_quartile' for col in cols_order] +
             [col + '_median' for col in cols_order] +
             [col + '_third_quartile' for col in cols_order] + 
             [col + '_argmin' for col in cols_order] + 
             [col + '_argmax' for col in cols_order] + 
             [col + '_range' for col in cols_order] +
             [col + '_lower_iqr' for col in cols_order] + 
             [col + '_upper_iqr' for col in cols_order] + 
             [col + '_iqr' for col in cols_order] + 
             [col + '_kurtosis' for col in cols_order] +
             [col + '_slope' for col in cols_order] + 
             [col + '_intercept' for col in cols_order] +
             [col + '_p_value' for col in cols_order] +
             [col + '_std_err' for col in cols_order] +
             [col + '_r_squared' for col in cols_order] + 
             [col + '_skew' for col in cols_order] +
             audio_col_names +
             ['Emotion'])

feature_matrix = pd.DataFrame(rows, columns = col_names)
feature_matrix = feature_matrix.dropna(axis=0)

# print(feature_matrix.head(5))


KeyboardInterrupt: 

In [38]:
feature_matrix.shape

(7442, 1514)

# Model(s) Testing and Training

In [44]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

X = feature_matrix.drop(columns = ["Emotion"])
y = feature_matrix["Emotion"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

dependent_model = SVC(kernel = 'rbf', C = 10, gamma = 'scale')

dependent_model.fit(X_train_scaled, y_train)
y_pred = dependent_model.predict(X_test_scaled)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.8072531900604433
              precision    recall  f1-score   support

         ANG       0.88      0.90      0.89       254
         DIS       0.78      0.80      0.79       254
         FEA       0.74      0.71      0.73       254
         HAP       0.96      0.92      0.94       255
         NEU       0.81      0.84      0.82       218
         SAD       0.67      0.67      0.67       254

    accuracy                           0.81      1489
   macro avg       0.81      0.81      0.81      1489
weighted avg       0.81      0.81      0.81      1489



In [40]:
# To find best parameters
param_grid = {
    'C': [0.1, 1, 10, 100, 1000],
    'gamma': [1, 0.1, 0.01, 0.001, 'scale'],
    'kernel': ['linear', 'rbf']
}
grid_search = GridSearchCV(estimator=SVC(kernel = 'rbf'), param_grid=param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)

grid_search.fit(X_train_scaled, y_train)
y_pred_gs = grid_search.predict(X_test_scaled)
print(accuracy_score(y_test, y_pred_gs))

print("Best Paramaters: ", grid_search.best_params_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
0.8079247817327065
Best Paramaters:  {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}


### Building the Independent Model

In [43]:
feature_matrix["Actor"] = [csv.split('_')[0] for csv in list_of_csvs]
list_actors = feature_matrix["Actor"].unique()
train_actors = list_actors[:73]
test_actors = list_actors[73:]

train_df = feature_matrix[feature_matrix["Actor"].isin(train_actors)]
test_df = feature_matrix[feature_matrix["Actor"].isin(test_actors)]

X_train_indep = train_df.drop(columns = ["Actor", "Emotion"])
y_train_indep = train_df["Emotion"]

X_test_indep = test_df.drop(columns=["Actor", "Emotion"])
y_test_indep = test_df["Emotion"]

indep_scaler = StandardScaler()
X_train_indep_scaled = indep_scaler.fit_transform(X_train_indep)
X_test_indep = indep_scaler.transform(X_test_indep)

independent_model = SVC(kernel = 'rbf', gamma = 'scale', C = 10)
independent_model.fit(X_train_indep_scaled, y_train_indep)
y_pred_indep = independent_model.predict(X_test_indep)
print(accuracy_score(y_test_indep, y_pred_indep))
print(classification_report(y_test_indep, y_pred_indep))

0.7572881355932204
              precision    recall  f1-score   support

         ANG       0.78      0.84      0.81       252
         DIS       0.76      0.77      0.76       252
         FEA       0.70      0.66      0.68       252
         HAP       0.90      0.90      0.90       252
         NEU       0.77      0.74      0.76       215
         SAD       0.64      0.63      0.63       252

    accuracy                           0.76      1475
   macro avg       0.76      0.76      0.76      1475
weighted avg       0.76      0.76      0.76      1475



In [47]:
import joblib

joblib.dump(dependent_model, "dependent_model.pkl")
joblib.dump(independent_model, "independent_model.pkl")
joblib.dump(scaler, "dependent_scaler.pkl")
joblib.dump(indep_scaler, "independent_scaler.pkl")

['independent_scaler.pkl']